# ADRec Dataset Experiments

Run independent full-training experiments for seeds 42, 43, 44, 45, and 46. Each seed trains its own item embeddings and ADRec model. A fresh batch trains all five seeds. Existing local checkpoint folders are not modified.

Before running:

1. Import this notebook into Kaggle.
2. Attach the updated `cmpe257_kaggle_code.zip` and `dataset.pkl`.
3. Enable Internet for package installation and select a GPU.

Run the numbered sections in order. Source ZIPs and extracted source folders are both supported.

Select `DATASET` in section 1. Each dataset uses its own weights and outputs. Start a fresh kernel when changing the dataset or code bundle. Multiple attached `dataset.pkl` files are resolved by their recorded checksum.


## 1. Load the project files

Locate the code and dataset under `/kaggle/input`. Extract or copy the code into `/kaggle/working/CMPE257_ML`, verify the source hashes, and add the project to the Python import path.

Leave the path variables as `None` for automatic discovery. When multiple matching inputs are attached, set `CODE_ZIP` or `CODE_DIR`, and `DATA_FILE`, to the required paths.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, zipfile

# None enables automatic discovery within the attached Kaggle inputs.
# Set explicit paths when multiple matching files are attached.
CODE_ZIP = None  # Path to the attached code ZIP
CODE_DIR = None  # Optional extracted folder containing bundle_manifest.json
DATASET = 'baby'  # 'baby', 'beauty', or 'ml-100k'
DATA_FILE = None  # Optional path to the selected dataset.pkl
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

def attached_file(name, override):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}; found {matches}. Set its path above.')
    return matches[0]

REPO = WORK_ROOT / 'CMPE257_ML'
REPO.mkdir(parents=True, exist_ok=True)
zip_matches = sorted(INPUT_ROOT.rglob('cmpe257_kaggle_code.zip'))
archive = None
if CODE_DIR is None and (CODE_ZIP is not None or zip_matches):
    archive = attached_file('cmpe257_kaggle_code.zip', CODE_ZIP)
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            if not (REPO / member).resolve().is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe archive member')
        z.extractall(REPO)
else:
    # Kaggle may unpack the uploaded ZIP before attaching it to a notebook.
    manifest_path = (Path(CODE_DIR) / 'bundle_manifest.json' if CODE_DIR
                     else attached_file('bundle_manifest.json', None))
    source_dir = manifest_path.parent.resolve()
    manifest = json.loads(manifest_path.read_text())
    for name in [*manifest['files'], 'bundle_manifest.json']:
        source, target = (source_dir / name).resolve(), (REPO / name).resolve()
        if not source.is_relative_to(source_dir) or not target.is_relative_to(REPO.resolve()):
            raise ValueError('Unsafe manifest path')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
manifest = json.loads((REPO / 'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    file = (REPO / name).resolve()
    if not file.is_relative_to(REPO.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source branch:', manifest['branch'])
print('Base commit:', manifest['base_commit'])
if archive:
    print('Source ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
else:
    print('Loaded Kaggle-extracted source; all manifest hashes verified.')
from src.data.catalog import find_dataset, dataset_spec
DATASET_PATH = find_dataset(INPUT_ROOT, DATASET, DATA_FILE)
ARCHIVE_ROOT = WORK_ROOT if DATASET == 'baby' else WORK_ROOT / 'exports' / DATASET
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET, DATASET_PATH)


## 2. Install dependencies

Install the packages listed in `requirements-shared.txt`. The installation retains the existing CUDA-enabled PyTorch package when it satisfies the requirements. Complete this cell before importing the models.

If installation requests a kernel restart, restart the session and rerun the setup and installation cells.


In [ ]:
import subprocess
# Keep Kaggle's CUDA-enabled torch if it satisfies the shared requirements.
# Internet must be enabled for missing dependencies. Run before importing models.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade-strategy', 'only-if-needed',
                '-r', str(REPO / 'requirements-shared.txt')], check=True)


## 3. Check the runtime

Import the shared data, training, and inference functions. Check the printed PyTorch version and GPU information. Full training requires `CUDA available: True`; smoke runs also support CPU execution.


In [ ]:
import torch
from src.data import load_dataset, SequenceDataset
from src.data.load import dataset_summary
from src.training.kaggle import make_config, train, load_for_inference, predict_scores
from src.utils.reproducibility import seed_everything

print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(1)
    print('CPU is sufficient for smoke mode. Enable a GPU before full training.')
# Check model imports before starting training.
from sasrec import SASRec
from model import Att_Diffuse_model


## 4. Configure the seed runs

`SEEDS` lists the five experiments. A fresh execution trains seeds 42–46. Saved stages are reused only when present in the current output directory or supplied through `RESUME_ROOT`. Full mode uses all users from the selected dataset, hidden size 128, 32 diffusion steps, and up to 500 epochs per stage with early stopping.

Each seed is applied to both embedding pretraining and ADRec training. The dataset splits and evaluation settings remain fixed. Runs are saved separately as `runs/adrec_full_seed<N>/`.

Set `RUN_MODE = 'smoke'` for a nine-user pipeline check. Smoke outputs use separate folders and are never combined with full results. Set `RESUME_ROOT` only when attaching previously saved run folders from another session.


In [ ]:
RUN_MODE = 'full'
SEEDS = [42, 43, 44, 45, 46]
RESUME_ROOT = None  # Folder containing saved adrec_full_seed<N> run directories

assert RUN_MODE in ('full', 'smoke')
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert all(type(seed) is int and seed in (42, 43, 44, 45, 46) for seed in SEEDS)
if RUN_MODE == 'full' and not torch.cuda.is_available():
    raise RuntimeError('Full training requires a GPU.')
if RESUME_ROOT is not None and not Path(RESUME_ROOT).is_dir():
    raise FileNotFoundError(RESUME_ROOT)

RUNS_ROOT = WORK_ROOT / 'runs'
if DATASET != 'baby':
    RUNS_ROOT /= DATASET
RESULTS_DIR = WORK_ROOT / ('results' if RUN_MODE == 'full' else 'smoke_results')
if DATASET != 'baby':
    RESULTS_DIR /= DATASET
RESULTS_DIR /= 'adrec'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg = make_config('adrec', RUN_MODE, dataset=DATASET)
data = load_dataset(DATASET_PATH, cfg['dataset_sha256'])
print('New seeds:', SEEDS)
print('Mode:', RUN_MODE)
print('Results directory:', RESULTS_DIR)
print(dataset_summary(data))
print(json.dumps(cfg, indent=2))


## 5. Define the result exports

The functions in the following cell copy per-seed metadata, record checkpoint hashes, and write the result CSVs. They run within this notebook and do not train models.

The CSV column names match the baseline experiments. Summaries use the completed seeds only and report arithmetic means and sample standard deviations. Configuration checks prevent mixing smoke and full runs or replacing a seed with conflicting results.


In [ ]:
import csv
import hashlib
import io
import json
import math
from pathlib import Path
import shutil
import statistics
from src.data.catalog import dataset_spec


METRICS = ["HR@5", "HR@10", "HR@20", "NDCG@5", "NDCG@10", "NDCG@20"]
RUN_COLUMNS = ["run_id", "model", "dataset", "seed", "best_epoch", "epochs_completed", *METRICS]
SUMMARY_COLUMNS = ["model", "dataset", "n_runs", "seeds", "statistic", *METRICS]
METADATA = ["config.json", "results.json", "history.json", "environment.json", "requirements-lock.txt"]


def _json(path):
    return json.loads(Path(path).read_text())


def _write(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text)
    temporary.replace(path)


def _hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _validate(cfg, result):
    if cfg["model"] != "adrec" or result["model"] != "adrec":
        raise ValueError("Expected ADRec results")
    if result["mode"] != cfg["mode"] or type(cfg["seed"]) is not int:
        raise ValueError("Invalid result mode or seed")
    test = result.get("test")
    if not test or test["protocol"] != "adrec_upstream" or test["units"] != "percent":
        raise ValueError("Export requires completed test evaluation using adrec_upstream")
    for key in METRICS:
        if not math.isfinite(test["metrics"][key]) or not 0 <= test["metrics"][key] <= 100:
            raise ValueError("Invalid test metric")
    return {k: v for k, v in cfg.items() if k != "seed"}


def _records(folder):
    records = []
    reference = None
    for path in Path(folder).glob("adrec_seed*_results.json"):
        prefix = path.name.removesuffix("results.json")
        cfg = _json(path.with_name(prefix + "config.json"))
        result = _json(path)
        if prefix != f"adrec_seed{cfg['seed']}_":
            raise ValueError("Seed does not match the filename")
        identity = _validate(cfg, result)
        if reference is not None and identity != reference:
            raise ValueError("Do not combine different configurations or smoke/full runs")
        reference = identity
        records.append((cfg, result))
    return sorted(records, key=lambda pair: pair[0]["seed"])


def summarize(folder):
    """Same CSV columns and sample-standard-deviation convention as the baselines."""
    folder = Path(folder)
    records = _records(folder)
    if not records:
        return []
    dataset = records[0][0].get("dataset", "baby")
    label = dataset_spec(dataset)["label"]
    rows = [{"run_id": f"adrec_{cfg['mode']}_seed{cfg['seed']}", "model": "ADRec",
             "dataset": label, "seed": cfg["seed"], "best_epoch": result["best_epoch"],
             "epochs_completed": result["epochs_completed"], **result["test"]["metrics"]}
            for cfg, result in records]
    seeds = [r["seed"] for r in rows]
    seed_label = (f"{seeds[0]}-{seeds[-1]}" if len(seeds) > 1 and seeds == list(range(seeds[0], seeds[-1] + 1))
                  else ",".join(map(str, seeds)))
    common = {"model": "ADRec", "dataset": label, "n_runs": len(rows), "seeds": seed_label}
    summary = [{**common, "statistic": "mean", **{k: round(statistics.mean(r[k] for r in rows), 4) for k in METRICS}}]
    if len(rows) > 1:
        summary.append({**common, "statistic": "std (sample, n-1)",
                        **{k: round(statistics.stdev(r[k] for r in rows), 4) for k in METRICS}})
    for filename, columns, values in [(f"adrec_{dataset}_runs.csv", RUN_COLUMNS, rows),
                                      (f"adrec_{dataset}_summary.csv", SUMMARY_COLUMNS, summary)]:
        stream = io.StringIO(newline="")
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(values)
        _write(folder / filename, stream.getvalue())
    cfg = records[0][0]
    coverage = "Complete five-seed set." if seeds == [42, 43, 44, 45, 46] else "Partial seed set; summary includes only the listed completed runs."
    _write(folder / "EXPERIMENT.md", f"""# Experiment record: ADRec on {label}

- Model: ADRec, {cfg['mode']} mode.
- Completed seeds: {seed_label}. {coverage}
- Dataset SHA-256: `{cfg['dataset_sha256']}`. Released train/validation/test splits retained.
- Training: separate embedding pretraining and ADRec training for each seed; the same seed is set in both stages.
- Training notebook: `notebooks/adrec_kaggle.ipynb`.
- Configuration: `adrec_seed<N>_config.json`; pretraining metadata is under `pretrain/`.
- Selection: improvement in any tracked validation metric; evaluation interval {cfg['eval_interval']}, patience {cfg['patience']}.
- Evaluation: `adrec_upstream`, batch size {cfg['evaluation_batch_size']}, HR/NDCG percentages, all catalog columns including padding; no seen-item filtering.
- Aggregation: arithmetic mean and sample standard deviation (n-1) of the per-seed test percentages. A single run has no reported standard deviation.
- Source/runtime: per-seed environment files contain source hashes, Python, PyTorch, CUDA, and hardware. Per-seed requirements files record installed packages.
- Checkpoints: hashes and storage references are in `adrec_seed<N>_checkpoints.json`. Weights are stored separately from these result files.
- Timing: training and inference benchmarks were not recorded by this export.
- Deviations: project seed settings, resumable training, enforced embedding freeze during alignment, and evaluation-mode checkpoint inference are documented in `docs/KAGGLE_TRAINING.md`.

`environment.json` and `requirements-lock.txt` are copies from the lowest listed seed for compatibility with the baseline result layout. Per-seed files retain each run's own environment.
""")
    first = seeds[0]
    for name in ("environment.json", "requirements-lock.txt"):
        source = folder / f"adrec_seed{first}_{name}"
        if source.exists():
            shutil.copyfile(source, folder / name)
    return rows


def export_run(run_dir, results_dir, checkpoint_location=None):
    """Copy metadata only; never include .pt weights in the shareable folder."""
    run_dir, results_dir = Path(run_dir), Path(results_dir)
    adrec, pretrain = run_dir / "adrec", run_dir / "pretrain"
    cfg, result = _json(adrec / "config.json"), _json(adrec / "results.json")
    identity = _validate(cfg, result)
    seed = cfg["seed"]
    pre_cfg = _json(pretrain / "config.json")
    expected_pre = dict(cfg, model="pretrain")
    if pre_cfg != expected_pre:
        raise ValueError("Pretraining and ADRec must use matching settings and seeds")
    for existing_cfg, existing_result in _records(results_dir):
        if _validate(existing_cfg, existing_result) != identity:
            raise ValueError("Existing export has incompatible configuration")
        if existing_cfg["seed"] == seed and existing_result != result:
            raise ValueError("A different result already exists for this seed")
    for stage in (pretrain, adrec):
        for name in [*METADATA, "best.pt", "latest.pt"]:
            if not (stage / name).is_file():
                raise FileNotFoundError(stage / name)
    for stage, source, target in [("pretrain", pretrain, results_dir / "pretrain"),
                                   ("adrec", adrec, results_dir)]:
        target.mkdir(parents=True, exist_ok=True)
        for name in METADATA:
            shutil.copyfile(source / name, target / f"{stage}_seed{seed}_{name}")
    checkpoints = {"seed": seed, "location": checkpoint_location or f"runs/{run_dir.name}",
                   "files": {f"{stage}/{name}": _hash(run_dir / stage / name)
                             for stage in ("pretrain", "adrec") for name in ("best.pt", "latest.pt")}}
    _write(results_dir / f"adrec_seed{seed}_checkpoints.json", json.dumps(checkpoints, indent=2) + "\n")
    return summarize(results_dir)


## 6. Train and export seeds 42–46

For each seed, run the following stages in sequence:

1. Train the original pretraining model with cross-entropy loss and select its checkpoint using validation.
2. Load that seed's pretrained item embeddings into ADRec.
3. Keep the embeddings frozen for five full-training epochs, then train the embeddings and backbone together.
4. Select the ADRec checkpoint using validation and evaluate it on the test set.
5. Export the result files, update the CSV summaries, and create a checkpoint ZIP for the completed seed.

The loop below creates both configurations for each seed and calls the shared `train()` function directly for pretraining and ADRec. Resume checks and exports are handled in the same notebook cell. Each completed epoch saves `latest.pt`. Rerunning the cell reuses completed stages and resumes interrupted stages found in the same output directory. Completed seeds are not trained again. A new Kaggle session requires saved run folders attached through `RESUME_ROOT`.

The output files contain actual completed results only. Each row contains the result from its own trained model.


In [ ]:
completed_runs = {}
results_archive = None

for seed in SEEDS:
    run_id = f'adrec_{RUN_MODE}_seed{seed}'
    run_dir = RUNS_ROOT / run_id
    prior_run = Path(RESUME_ROOT) / run_id if RESUME_ROOT else None

    pretrain_cfg = make_config('pretrain', RUN_MODE, dataset=DATASET)
    adrec_cfg = make_config('adrec', RUN_MODE, dataset=DATASET)
    pretrain_cfg['seed'] = seed
    adrec_cfg['seed'] = seed

    # Pretrain embeddings first, then initialize ADRec from this seed's best embeddings.
    for stage, stage_cfg in [('pretrain', pretrain_cfg), ('adrec', adrec_cfg)]:
        stage_dir = run_dir / stage
        source_dir = stage_dir if (stage_dir / 'latest.pt').exists() else None
        if source_dir is None and prior_run and (prior_run / stage / 'latest.pt').exists():
            source_dir = prior_run / stage

        resume_checkpoint = None
        if source_dir is not None:
            saved_cfg = json.loads((source_dir / 'config.json').read_text())
            if saved_cfg != stage_cfg:
                raise ValueError(f'Stage configuration mismatch: {source_dir}')
            report_path = source_dir / 'results.json'
            saved_result = json.loads(report_path.read_text()) if report_path.exists() else None
            complete = saved_result and (saved_result['early_stopped'] or
                                         saved_result['epochs_completed'] >= stage_cfg['epochs'])
            evaluated = stage == 'pretrain' or saved_result and saved_result.get('test') is not None
            if complete and evaluated and (source_dir / 'best.pt').exists():
                if source_dir.resolve() != stage_dir.resolve():
                    shutil.copytree(source_dir, stage_dir, dirs_exist_ok=True)
                print(f'Reusing completed {stage}, seed {seed}')
                continue
            resume_checkpoint = source_dir / 'latest.pt'

        print(f'Training {stage}, seed {seed}, mode {RUN_MODE}', flush=True)
        pretrained_checkpoint = run_dir / 'pretrain' / 'best.pt' if stage == 'adrec' else None
        train(stage_cfg, DATASET_PATH, stage_dir,
              pretrained_checkpoint=pretrained_checkpoint,
              resume=resume_checkpoint, finalize=stage == 'adrec')

    result = json.loads((run_dir / 'adrec' / 'results.json').read_text())
    rows = export_run(run_dir, RESULTS_DIR)
    completed_runs[seed] = {'run_dir': str(run_dir), 'result': result, 'rows': rows}

    checkpoint_archive = shutil.make_archive(str(ARCHIVE_ROOT / run_id), 'zip', root_dir=run_dir)
    archive_name = 'adrec_results' if RUN_MODE == 'full' else 'adrec_smoke_results'
    results_archive = shutil.make_archive(str(ARCHIVE_ROOT / archive_name), 'zip', root_dir=RESULTS_DIR)
    print('Completed seed:', seed)
    print('Checkpoint archive:', checkpoint_archive)
    print('Results archive:', results_archive)
    print(json.dumps(result['test'], indent=2))


## 7. Inspect the results and summary

`adrec_<dataset>_runs.csv` contains one row per seed with the selected epoch and all six test metrics. `adrec_<dataset>_summary.csv` uses the same columns as the baseline summaries and reports the mean and sample standard deviation (`n-1`). Values are percentages rounded to four decimal places. A single completed run has no standard-deviation row.

A complete comparison contains seeds 42–46 and `n_runs = 5`. Variation across seeds measures sensitivity to random initialization, training order, dropout, and diffusion noise. It does not measure variation across different datasets.


In [ ]:
rows = summarize(RESULTS_DIR)
print('Completed seeds:', [row['seed'] for row in rows])
print('Per-seed results:')
print((RESULTS_DIR / f'adrec_{DATASET}_runs.csv').read_text())
print('Summary:')
print((RESULTS_DIR / f'adrec_{DATASET}_summary.csv').read_text())

archive_name = 'adrec_results' if RUN_MODE == 'full' else 'adrec_smoke_results'
results_archive = shutil.make_archive(str(ARCHIVE_ROOT / archive_name), 'zip', root_dir=RESULTS_DIR)
print('Results archive:', results_archive)


## 8. Load a saved model and generate recommendations

Load the first completed seed's `best.pt` and predict ten item IDs from the visible validation history. The held-out target is excluded. Padding ID 0 is removed from this inference example; benchmark evaluation retains its original candidate rules.


In [ ]:
INFERENCE_SEED = SEEDS[0]
MODEL_DIR = RUNS_ROOT / f'adrec_{RUN_MODE}_seed{INFERENCE_SEED}' / 'adrec'
model, saved_cfg = load_for_inference(MODEL_DIR / 'best.pt')
example = SequenceDataset(data, 'val', saved_cfg['max_len'])[0]
seed_everything(saved_cfg['seed'])
scores = predict_scores(model, [example['input_ids']]).cpu()
scores[:, 0] = -torch.inf
recommended_ids = torch.topk(scores, k=10, dim=-1).indices[0].tolist()
print('Seed:', INFERENCE_SEED)
print('Visible history:', [item for item in example['input_ids'] if item])
print('Recommended product IDs:', recommended_ids)


## 9. Download the output files

Download `adrec_results.zip` and each `adrec_full_seed<N>.zip` from Kaggle's `/kaggle/working` output file browser before ending the session. The ZIP files are created after each completed seed.

Extract `adrec_results.zip` into the repository's `results/adrec/` folder. It contains:

- `adrec_seed<N>_results.json`, `adrec_seed<N>_config.json`, and `adrec_seed<N>_history.json`.
- Per-seed environment and requirements files, plus baseline-compatible `environment.json` and `requirements-lock.txt` copies from the lowest listed seed.
- `adrec_<dataset>_runs.csv`, `adrec_<dataset>_summary.csv`, and `EXPERIMENT.md`.
- Checkpoint SHA-256 records and the pretraining metadata under `pretrain/`.

The result archive contains no weights. Extract each checkpoint ZIP into `checkpoints/adrec_full_seed<N>/`; it contains that seed's `pretrain/` and `adrec/` folders, including `best.pt` and `latest.pt`. Keep previous experiment folders separate from the new batch.

Kaggle **Save & Run All** starts a separate execution from the first cell. Background execution uses the attached inputs and the configured resume folders; it does not inherit the current interactive session's files.

Baby keeps the original paths above. For Beauty or MovieLens, ZIPs are under `/kaggle/working/exports/<dataset>/`. Extract result ZIPs into `results/<dataset>/adrec/` and model ZIPs into `checkpoints/<dataset>/adrec_full_seed<N>/`. Working runs are under `runs/<dataset>/`. Keep dataset folders separate when downloading.


## 10. Resume interrupted experiments

1. Save the existing `runs/adrec_full_seed<N>/` folders before ending the current session, including any interrupted stage's `latest.pt` and metadata.
2. Attach the saved folders in the next Kaggle session.
3. Set `RESUME_ROOT` to their parent folder. The expected layout is `<RESUME_ROOT>/adrec_full_seed42/pretrain/` and `<RESUME_ROOT>/adrec_full_seed42/adrec/`, with the same layout for the other saved seeds.
4. Retain the original run mode and configuration. Run the notebook from the setup section.

Completed stages with matching configurations are copied to the working directory and reused. Interrupted stages resume from their latest completed epoch. Seeds without saved stages start from scratch. Early-stopped stages remain stopped. Hardware or library changes can affect numerical reproducibility.
